# Fairness Monitoring Module — Demo

End-to-end demonstration using the **loan approval dataset**.

**Sensitive attribute:** `Gender`  
**Target:** `Loan_Approval_Status`  
**Model:** Logistic Regression (trained here from scratch)

Pipeline:
1. Load data & train model
2. `RealTimeFairnessTracker` — simulate production stream in batches
3. `FairnessDriftAndAlertEngine` — detect fairness drift
4. `FairnessReportingDashboard` — visualize & report
5. `FairnessABTestAnalyzer` — compare baseline vs fair model


## 0. Imports

In [45]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from datetime import datetime, timedelta

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

from tracker    import RealTimeFairnessTracker
from drift      import FairnessDriftAndAlertEngine
from reporting  import FairnessReportingDashboard
from ab_testing import FairnessABTestAnalyzer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports OK")


Imports OK


---
## 1. Load Data & Train Model


In [46]:
df = pd.read_csv("../data/loan_dataset.csv")
df.dropna(inplace=True)
print(f"Dataset shape : {df.shape}")
print(f"Approval rate : {df['Loan_Approval_Status'].mean():.3f}")
print(f"Gender counts :\n{df['Gender'].value_counts()}")


Dataset shape : (52000, 27)
Approval rate : 0.642
Gender counts :
Gender
Male      26011
Female    25989
Name: count, dtype: int64


In [47]:
FEATURE_COLS = ["Age", "Annual_Income", "Credit_Score",
                "Loan_Amount_Requested", "Employment_Status"]
TARGET_COL   = "Loan_Approval_Status"
SENSITIVE    = "Gender"

df_enc = pd.get_dummies(df[FEATURE_COLS + [TARGET_COL, SENSITIVE]], drop_first=True)

X = df_enc.drop(columns=[TARGET_COL])
y = df_enc[TARGET_COL]
gender_col = [c for c in X.columns if "Gender" in c][0]  # e.g. "Gender_Male"

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train_sc, y_train)
y_pred = model.predict(X_test_sc)
y_proba = model.predict_proba(X_test_sc)[:, 1]

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.85      0.71      0.77      3727
           1       0.85      0.93      0.89      6673

    accuracy                           0.85     10400
   macro avg       0.85      0.82      0.83     10400
weighted avg       0.85      0.85      0.85     10400



In [48]:
# Recover Gender for the test set (0 = Female, 1 = Male based on get_dummies drop_first)
gender_binary = X_test[gender_col].values  # 1 = Male, 0 = Female

print("Approval rate by gender (predictions):")
for g, label in [(1, "Male"), (0, "Female")]:
    mask = gender_binary == g
    print(f"  {label}: {y_pred[mask].mean():.3f}  (n={mask.sum()})")


Approval rate by gender (predictions):
  Male: 0.701  (n=5203)
  Female: 0.705  (n=5197)


---
## 2. RealTimeFairnessTracker

We simulate a **production stream** by shuffling the test set and
splitting it into 25 equal batches ingested one at a time.
Each batch represents one hour of production traffic.


In [49]:
# Shuffle test set and split into batches
N_BATCHES  = 25
rng        = np.random.default_rng(RANDOM_STATE)
idx        = rng.permutation(len(y_test))

predictions_all = y_pred[idx]
labels_all      = y_test.values[idx]
sensitive_all   = gender_binary[idx]

batch_size = len(idx) // N_BATCHES
batches = [
    {
        "predictions": predictions_all[i*batch_size:(i+1)*batch_size],
        "labels":      labels_all     [i*batch_size:(i+1)*batch_size],
        "sensitive":   sensitive_all  [i*batch_size:(i+1)*batch_size],
    }
    for i in range(N_BATCHES)
]

print(f"Total test samples : {len(idx)}")
print(f"Batches            : {N_BATCHES}")
print(f"Samples per batch  : {batch_size}")


Total test samples : 10400
Batches            : 25
Samples per batch  : 416


In [50]:
tracker = RealTimeFairnessTracker(window_size=8, min_group_size=10)

BASE_TS = datetime(2024, 6, 1, 9, 0)

for i, batch in enumerate(batches):
    ts = BASE_TS + timedelta(hours=i)
    tracker.ingest(**batch, timestamp=ts, sensitive_name="gender")

history = tracker.history
print(f"History shape : {history.shape}")
print(f"Columns       : {list(history.columns)}")
history.tail(6)


History shape : (50, 11)
Columns       : ['gender', 'group_size', 'small_group_warning', 'positive_rate', 'tpr', 'fpr', 'ppv', 'demographic_parity', 'equalized_odds', 'predictive_parity', 'window_batches']


,gender,group_size,small_group_warning,positive_rate,tpr,fpr,ppv,demographic_parity,equalized_odds,predictive_parity,window_batches
timestamp,,,,,,,,,,,
2024-06-02 07:00:00,False,1649,False,0.688296,0.928709,0.279869,0.849339,0.013908,0.006563,0.005623,8
2024-06-02 07:00:00,True,1679,False,0.702204,0.931608,0.286432,0.854962,0.013908,0.006563,0.005623,8
2024-06-02 08:00:00,False,1643,False,0.696896,0.931429,0.281619,0.854148,0.008149,0.012005,0.001455,8
2024-06-02 08:00:00,True,1685,False,0.705045,0.930211,0.293624,0.852694,0.008149,0.012005,0.001455,8
2024-06-02 09:00:00,False,1644,False,0.694647,0.933908,0.278333,0.853765,0.007253,0.013470,0.004358,8
2024-06-02 09:00:00,True,1684,False,0.701900,0.934823,0.291803,0.849408,0.007253,0.013470,0.004358,8


In [51]:
print("=== Latest window snapshot ===")
print(tracker.latest()[["gender", "demographic_parity", "equalized_odds",
                         "positive_rate", "group_size"]]
      .to_string(index=False))


=== Latest window snapshot ===
gender  demographic_parity  equalized_odds  positive_rate  group_size
 False            0.007253         0.01347       0.694647        1644
  True            0.007253         0.01347       0.701900        1684


---
## 3. FairnessDriftAndAlertEngine

Runs a KS test to detect drift and classifies it as `spike` or `trend`
via wavelet decomposition. Alerts are prioritized as `CRITICAL / HIGH / LOW`.


In [52]:
engine = FairnessDriftAndAlertEngine(
    reference_window=8,
    ks_alpha=0.05,
    adaptive=True,
    group_size_weight=True,
)

alerts = engine.analyze(history)
print(f"Alerts generated : {len(alerts)}")
for a in alerts:
    print(f"  [{a.severity:8s}] {a.metric:25s}  "
          f"KS={a.drift_statistic:.3f}  p={a.p_value:.4f}  "
          f"score={a.severity_score:.3f}  wavelet={a.wavelet_trend}")


Alerts generated : 3
  [CRITICAL] demographic_parity         KS=0.750  p=0.0003  score=1.000  wavelet=None
  [CRITICAL] equalized_odds             KS=0.321  p=0.4050  score=0.701  wavelet=None
  [CRITICAL] predictive_parity          KS=0.857  p=0.0000  score=1.000  wavelet=None


In [53]:
engine.alert_summary()


,timestamp,metric,group,severity,severity_score,drift_statistic,p_value,wavelet_trend,message
0,2024-06-02 09:00:00,demographic_parity,False,CRITICAL,1.000,0.7500,0.0003,None,[CRITICAL] Drift detected in 'demographic_pari...
1,2024-06-02 09:00:00,equalized_odds,False,CRITICAL,0.701,0.3214,0.4050,None,[CRITICAL] Drift detected in 'equalized_odds' ...
2,2024-06-02 09:00:00,predictive_parity,False,CRITICAL,1.000,0.8571,0.0000,None,[CRITICAL] Drift detected in 'predictive_parit...


In [54]:
# Adaptive threshold status
if engine.threshold_manager is not None:
    print("Current adaptive thresholds:")
    print(engine.threshold_manager.summary().to_string(index=False))


Current adaptive thresholds:
            metric  threshold  alerts_fired  false_positives  fp_rate
demographic_parity       0.05             0                0      0.0
    equalized_odds       0.07             0                0      0.0
 predictive_parity       0.07             0                0      0.0


---
## 4. FairnessReportingDashboard


In [55]:
dashboard = FairnessReportingDashboard(title="Loan Approval — Fairness Monitor")


### 4a. Demographic parity over time

In [56]:
fig = dashboard.trend_plot(
    history,
    metric="demographic_parity",
    group_col="gender",
    threshold=engine.get_threshold("demographic_parity"),
    alert_timestamps=[a.timestamp for a in alerts if a.metric == "demographic_parity"],
    title="Demographic Parity Gap Over Time (Male vs Female)",
)
fig.show()


### 4b. Positive rate by gender

In [57]:
fig = dashboard.trend_plot(
    history,
    metric="positive_rate",
    group_col="gender",
    title="Approval Rate by Gender Over Time",
)
fig.show()


### 4c. Intersectional bar chart

In [58]:
fig = dashboard.intersectional_plot(
    history,
    metric="positive_rate",
    attr1_col="gender",
    recent_n=8,
    title="Mean Approval Rate by Gender (last 8 batches)",
)
fig.show()


### 4d. Multi-metric summary

In [59]:
fig = dashboard.summary_plot(
    history,
    metrics=["demographic_parity", "equalized_odds", "predictive_parity"],
    group_col="gender",
)
fig.show()


### 4e. Markdown report

In [60]:
path = dashboard.generate_report(
    history=history,
    alerts=alerts,
    output_path="fairness_report.md",
    group_col="gender",
)
with open(path) as f:
    print(f.read())


# Loan Approval â€” Fairness Monitor
*Generated: 2026-05-22 11:57 UTC*

---

## Executive Summary

| | |
|---|---|
| **Batches processed** | 50 |
| **Active alerts** | 3 |
| **Critical alerts** | 3 |
| **High alerts** | 0 |

### â›” Status: CRITICAL â€” Immediate action required.

## Current Fairness Metrics

- **Demographic Parity**: `0.0073`
- **Equalized Odds**: `0.0135`
- **Predictive Parity**: `0.0044`

## Group Positive Rates (latest window)

| Group | Positive Rate | Group Size |
|---|---|---|
| False | 0.6961 | 1649 |
| True | 0.7020 | 1678 |

## Active Alerts

| Time | Metric | Severity | KS Stat | p-value | Message |
|---|---|---|---|---|---|
| 2024-06-02 09:00:00 | demographic_parity | **CRITICAL** | 0.7500 | 0.0003 | [CRITICAL] Drift detected in 'demographic_parity' (KS=0.750, p=0.0003, threshold=0.05). Wavelet: N/A. |
| 2024-06-02 09:00:00 | predictive_parity | **CRITICAL** | 0.8571 | 0.0000 | [CRITICAL] Drift detected in 'predictive_parity' (KS=0.857, p=0.0000, threshold=

---
## 5. FairnessABTestAnalyzer

We compare:
- **Control** — baseline logistic regression (no fairness constraint)
- **Treatment** — model trained with class weights to partially mitigate gender bias

This simulates evaluating whether a fairness intervention actually helped.


In [61]:
# Treatment model: class_weight='balanced' as a simple fairness intervention
model_fair = LogisticRegression(
    max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"
)
model_fair.fit(X_train_sc, y_train)
y_pred_fair = model_fair.predict(X_test_sc)

print("=== Baseline model ===")
print(f"  Overall accuracy : {(y_pred == y_test.values).mean():.4f}")
print(f"  Female approval  : {y_pred[gender_binary==0].mean():.4f}")
print(f"  Male approval    : {y_pred[gender_binary==1].mean():.4f}")
print()
print("=== Fair model (balanced weights) ===")
print(f"  Overall accuracy : {(y_pred_fair == y_test.values).mean():.4f}")
print(f"  Female approval  : {y_pred_fair[gender_binary==0].mean():.4f}")
print(f"  Male approval    : {y_pred_fair[gender_binary==1].mean():.4f}")


=== Baseline model ===
  Overall accuracy : 0.8506
  Female approval  : 0.7046
  Male approval    : 0.7009

=== Fair model (balanced weights) ===
  Overall accuracy : 0.8511
  Female approval  : 0.7037
  Male approval    : 0.7002


In [62]:
# Build DataFrames for the analyzer
ctrl_df = pd.DataFrame({
    "prediction" : y_pred,
    "label"      : y_test.values,
    "gender"     : np.where(gender_binary == 1, "Male", "Female"),
    # Continuous columns for mediation
    "score"      : y_proba,
    "approval_rate": y_pred.astype(float),
})

trt_df = pd.DataFrame({
    "prediction" : y_pred_fair,
    "label"      : y_test.values,
    "gender"     : np.where(gender_binary == 1, "Male", "Female"),
    "score"      : model_fair.predict_proba(X_test_sc)[:, 1],
    "approval_rate": y_pred_fair.astype(float),
})

analyzer = FairnessABTestAnalyzer(
    control=ctrl_df,
    treatment=trt_df,
    pred_col="prediction",
    label_col="label",
    sensitive_cols=["gender"],
    alpha=0.05,
)


### 5a. Statistical power per subgroup

In [63]:
power_df = analyzer.calculate_power(effect_size=0.05, metric="accuracy")
print("=== Power Analysis (can we detect a 5% accuracy difference per gender?) ===")
power_df


=== Power Analysis (can we detect a 5% accuracy difference per gender?) ===


,subgroup,n_control,n_treatment,power,baseline_rate,detectable_effect
0,Female,5197,5197,1.0,0.8495,0.05
1,Male,5203,5203,1.0,0.8516,0.05


### 5b. Heterogeneous treatment effects

In [64]:
hte_df = analyzer.heterogeneous_effects(
    business_metric="accuracy",
    fairness_metric="positive_rate",
)
print("=== Did the fair model help each gender equally? ===")
hte_df


=== Did the fair model help each gender equally? ===


,subgroup,n_control,n_treatment,accuracy_control,accuracy_treatment,accuracy_delta,accuracy_ci_low,accuracy_ci_high,positive_rate_control,positive_rate_treatment,positive_rate_delta,positive_rate_ci_low,positive_rate_ci_high,p_value,significant
0,Female,5197,5197,0.8495,0.8501,0.0006,-0.0140,0.0137,0.7046,0.7037,-0.0010,-0.0195,0.0158,0.9344,False
1,Male,5203,5203,0.8516,0.8520,0.0004,-0.0136,0.0142,0.7009,0.7002,-0.0008,-0.0174,0.0162,0.9560,False


### 5c. Mediation analysis

In [65]:
# Does the intervention work by changing approval rates (mediator),
# or does it have a direct effect on the outcome score?
mediation = analyzer.mediation_analysis(
    outcome_col="score",
    mediator_col="approval_rate",
    treatment_indicator_col="treatment",
)

print("=== Baron-Kenny Mediation Analysis ===")
print(f"  Total effect        : {mediation['total_effect']:+.4f}  (p={mediation['p_total']:.4f})")
print(f"  Direct effect       : {mediation['direct_effect']:+.4f}  (p={mediation['p_direct']:.4f})")
print(f"  Indirect effect     : {mediation['indirect_effect']:+.4f}")
prop = mediation['proportion_mediated']
if not (isinstance(prop, float) and prop != prop):  # nan check
    print(f"  Proportion mediated : {prop:.1%}")
else:
    print(f"  Proportion mediated : N/A")


=== Baron-Kenny Mediation Analysis ===
  Total effect        : -0.0866  (p=0.0000)
  Direct effect       : -0.0860  (p=0.0000)
  Indirect effect     : -0.0005
  Proportion mediated : 0.6%


---
## Summary

| Component | What was demonstrated |
|---|---|
| `RealTimeFairnessTracker` | 25-batch production stream from real loan test data, sliding window metrics per gender |
| `FairnessDriftAndAlertEngine` | KS-test drift detection + wavelet classification on real model predictions |
| `FairnessReportingDashboard` | Trend plots, approval rate by gender, intersectional bar chart, Markdown report |
| `FairnessABTestAnalyzer` | Baseline vs balanced-weight model: power analysis, HTE per gender, mediation |


---
## 6. Streamlit Interactive Dashboard

`app.py` provides a live interactive version of the monitoring module.
It uses synthetic batches so it can run standalone — no dataset needed.

Features:
- **Sidebar** — control drift level, batch size, pre-fill history, auto-ingest
- **Trends tab** — live fairness metric plots with alert markers
- **Intersectional tab** — bar chart per group + raw history table
- **Alerts tab** — alert table, adaptive threshold status, downloadable Markdown report
- **A/B Test tab** — interactive simulator with sliders for group size and drift

To run it, make sure `app.py` is at the project root (same level as the `monitoring/` folder):


In [67]:
# Run from terminal:
#   streamlit run app.py
#
# Or launch directly from the notebook (opens in browser):
import subprocess, sys
print("To launch the dashboard, run in your terminal:")
print()
print("   streamlit run app.py")
print()
print("The app will open at http://localhost:8501")
print()
print("Suggested demo flow:")
print("  1. Click 'Pre-fill history' (15 batches) to populate the tracker")
print("  2. Increase 'Bias Drift Level' to 0.20-0.30")
print("  3. Click 'Ingest one batch' several times and watch metrics update")
print("  4. Check the Alerts tab for drift detection results")
print("  5. Go to A/B Test tab and click Run A/B Analysis")


To launch the dashboard, run in your terminal:

   streamlit run app.py

The app will open at http://localhost:8501

Suggested demo flow:
  1. Click 'Pre-fill history' (15 batches) to populate the tracker
  2. Increase 'Bias Drift Level' to 0.20-0.30
  3. Click 'Ingest one batch' several times and watch metrics update
  4. Check the Alerts tab for drift detection results
  5. Go to A/B Test tab and click Run A/B Analysis


### How app.py connects to the monitoring module

`app.py` uses the same four classes from `monitoring/` as this notebook:

| Component | Usage in app.py |
|---|---|
| `RealTimeFairnessTracker` | Stored in `st.session_state`, ingests one batch per button click or auto-loop |
| `FairnessDriftAndAlertEngine` | Re-runs `analyze(history)` on every page render to catch new drift |
| `FairnessReportingDashboard` | Renders Plotly charts inline via `st.plotly_chart` |
| `FairnessABTestAnalyzer` | Instantiated on-demand when the user clicks Run A/B Analysis |

The main difference from this notebook: `app.py` uses synthetic batches
(`_simulate_batch`) to allow the dashboard to run standalone.
To connect it to real data you would replace `_simulate_batch`
with a function that pulls from your production database or the loan CSV.
